# AbstractTensor — Manifolds

This notebook tests and explains `manifolds.jl` and `indices.jl` step by step.

---
## 1. Loading package

In [1]:
using AbstractTensors

---
## 2. Manifolds and Vbundles 
---

### a. Manifold type and definition

#### Documentation access
- `@doc Manifold` — shows the docstring directly (julia macro)
- `?Manifold` — same, with search results prepended (Jupyter help mode)

In [2]:
@doc Manifold

```julia
Manifold
```

Struct representing a registered differentiable manifold. Instances are created by [`@def_manifold`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
M.name              # :M
M.dim               # 4 
M.tangent_bundle    # :tangentM
M.cotangent_bundle  # :cotangentM
M.vbundles          # [:TangentM, :CoTangentM]
```

### Fields

  * `name`             : manifold name, e.g. `:M`
  * `dim`              : dimension
  * `tangent_bundle`   : name of the tangent bundle, e.g. `:tangentM`
  * `cotangent_bundle` : name of the cotangent (dual) bundle, e.g. `:cotangentM`
  * `vbundles`         : all associated bundle names


In [3]:
?@def_manifold

```julia
@def_manifold name dim [idx1, idx2, ...]
```

Define a new manifold and automatically create its tangent and cotangent bundles. Bind the following variables in the caller's scope:

  * `name`            → a [`Manifold`](@ref) instance
  * `tangent<name>`   → a [`VBundle`](@ref) instance (`isdual = false`)
  * `cotangent<name>` → a [`VBundle`](@ref) instance (`isdual = true`)

Each index symbol is also bound to an [`IndexSymbol`](@ref) in the caller's scope.

`dim` can be a concrete integer or a symbolic name for parametric manifolds:

Register `name` in `_MANIFOLDS`, the tangent and cotangent bundles in `_VBUNDLES`, and all index symbols in `_INDICES`.

#### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]   # concrete dimension
@def_manifold M d [b1, b2, b3, b4]  # parametric dimension
```


In [4]:
@def_manifold M 2 [a1, a2, a3, a4]

Defined manifold M of dimension 2 with tangent bundle tangentM and cotangent bundle cotangentM


In [5]:
@show M.name
@show M.dim
@show M.tangent_bundle
@show M.cotangent_bundle
@show M.vbundles;

M.name = :M
M.dim = 2
M.tangent_bundle = :tangentM
M.cotangent_bundle = :cotangentM
M.vbundles = [:tangentM, :cotangentM]


In [6]:
@def_manifold N d [b1, b2, b3, b4]

Defined manifold N of dimension d with tangent bundle tangentN and cotangent bundle cotangentN


In [7]:
# Registries
display(_MANIFOLDS)
display(_VBUNDLES)
display(_INDICES)

Dict{Symbol, Manifold} with 2 entries:
  :N => Manifold(N, dim=d, TBundle=tangentN, CBundle=cotangentN)
  :M => Manifold(M, dim=2, TBundle=tangentM, CBundle=cotangentM)

Dict{Symbol, VBundle} with 4 entries:
  :cotangentM => VBundle(cotangentM, cotangent, manifold=M, dim=2)
  :tangentM   => VBundle(tangentM, tangent, manifold=M, dim=2)
  :cotangentN => VBundle(cotangentN, cotangent, manifold=N, dim=d)
  :tangentN   => VBundle(tangentN, tangent, manifold=N, dim=d)

Dict{Symbol, Symbol} with 8 entries:
  :a2 => :tangentM
  :b4 => :tangentN
  :a3 => :tangentM
  :b2 => :tangentN
  :a4 => :tangentM
  :b1 => :tangentN
  :a1 => :tangentM
  :b3 => :tangentN

In [8]:
M

Dimension,2
Tangent Bundle,tangentM
Cotangent Bundle,cotangentM
All VBundles,"tangentM, cotangentM"


In [9]:
@undef_manifold M

┌ Warning: Manifold M has been undefined. The variable M still holds a stale reference, accessing it will error. Restart the kernel to fully clear the binding.
└ @ Main ~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/manifolds.jl:382


Undefined tangent bundle:   tangentM
Undefined cotangent bundle: cotangentM


In [10]:
M.name
M.dim
M.tangent_bundle
M.cotangent_bundle
M.vbundles

┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/manifolds.jl:421
┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/manifolds.jl:421
┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/manifolds.jl:421
┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/manifolds.jl:421
┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/manifolds.jl:421


In [11]:
@def_manifold M 4 [a1, a2, a3, a4]

Defined manifold M of dimension 4 with tangent bundle tangentM and cotangent bundle cotangentM


In [12]:
@show M.name
@show M.dim
@show M.tangent_bundle
@show M.cotangent_bundle
@show M.vbundles;

M.name = :M
M.dim = 4
M.tangent_bundle = :tangentM
M.cotangent_bundle = :cotangentM
M.vbundles = [:tangentM, :cotangentM]


In [61]:
manifold_info(M)

  Manifold: M
  Dimension:        4
  Tangent bundle:   tangentM
  Cotangent bundle: cotangentM
  Vbundles:         [:tangentM, :cotangentM, :E, :dualE]
  Index Symbols:    [:a1, :a2, :a3, :a4]


### b. Tangent and Cotangent bundles

In [13]:
@doc VBundle

```julia
VBundle
```

Struct representing a registered vector bundle. Instances are created by [`@def_manifold`](@ref)  for the tangent and cotangent bundles,  and bound to variables in the caller's scope (e.g. `tangentM`, `cotangentM`).

Provides dot access to all metadata:

```julia
tangentM.name      # :tangentM
tangentM.manifold  # :M
tangentM.dim       # 4
tangentM.isdual    # false
tangentM.indices   # [TensorIndex(:a1, :TangentM), ...]
```

### Fields

  * `name`     : bundle name, e.g. `:TangentM`
  * `manifold` : base manifold name, e.g. `:M`
  * `dim`      : fibre dimension
  * `isdual`   : false = tangent (standard) bundle, true = cotangent (dual) bundle.              This is the single authoritative source of bundle variance —              no naming convention is relied upon.
  * `indices`  : the `TensorIndex` objects living in this bundle


In [14]:
tangentM

VBundle(tangentM, tangent, manifold=M, dim=4)

In [64]:
display(tangentM.name)
display(tangentM.manifold)
display(tangentM.dim)
display(tangentM.isdual)
display(tangentM.indices)

:tangentM

:M

4

false

4-element Vector{TensorIndex}:
  a1 ∈ tangentM
  a2 ∈ tangentM
  a3 ∈ tangentM
  a4 ∈ tangentM

In [65]:
indices_of(tangentM)

LoadError: MethodError: no method matching indices_of(::VBundle)
The function `indices_of` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  indices_of([91m::TensorExpression[39m)
[0m[90m   @[39m [35mAbstractTensors[39m [90m~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/[39m[90m[4mtensorExpressions.jl:179[24m[39m


### c. Definition of extra Vbundles

In [62]:
?@def_vbundle

```julia
@def_vbundle name manifold dim [idx1, idx2, ...]
```

Define a new vector bundle `name` of fibre dimension `dim` over `manifold`, and its dual bundle `dual<name>`. Bind the following variables in the caller's scope:

  * `name`       → a [`VBundle`](@ref) instance (`isdual = false`)
  * `dualname`   → a [`VBundle`](@ref) instance (`isdual = true`)

Each index symbol in `[idx1, idx2, ...]` is bound to an [`IndexSymbol`](@ref) and registered to `name` (the primal bundle). The dual bundle shares the same index symbols — `down(v1)` resolves to `TensorIndex(:v1, :dualname)` automatically via `dual_bundle`.

`dim` accepts a concrete integer or a bare symbol for parametric bundles. The fibre dimension is independent of the base manifold dimension.

Registers both bundles in `_VBUNDLES` and appends their names to `manifold.vbundles`.

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]
@def_vbundle E M 3 [A1, A2, A3]    # rank-3 bundle and its dual over M
@def_vbundle E M r [A1, A2, A3]    # parametric fibre dimension

E.isdual       # false
dualE.isdual   # true
v1.vbundle     # :E
up(A1)         # TensorIndex(:v1, :E)
down(A1)       # TensorIndex(:v1, :dualE)
M.vbundles     # [:tangentM, :cotangentM, :E, :dualE]
```


In [17]:
@def_vbundle E M 3 [A1, A2, A3]

Defined VBundle E (dim=3) and dual dualE over manifold M


In [56]:
E

VBundle(E, tangent, manifold=M, dim=3)

In [57]:
dualE

VBundle(dualE, cotangent, manifold=M, dim=3)

In [63]:
indices_of(E)

LoadError: MethodError: no method matching indices_of(::VBundle)
The function `indices_of` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  indices_of([91m::TensorExpression[39m)
[0m[90m   @[39m [35mAbstractTensors[39m [90m~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/[39m[90m[4mtensorExpressions.jl:179[24m[39m
